# Weekly retail demand forecasting

Forecast `units_sold` for the supplied 12-week test period using the same reusable implementation as the command line pipeline.

The notebook uses a fixed forecast origin: target-period sales never enter historical features. Future prices and promotion flags must be known when forecasting, as they are in the supplied test file. Run all cells after installing `.[dev]` and placing the CSVs in `data/`.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository or notebooks directory"
sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from demand_forecasting import DemandForecaster, evaluate, read_data

DATA = Path(os.environ.get("DEMAND_DATA_DIR", str(ROOT / "data")))
OUT = ROOT / "artifacts"
OUT.mkdir(exist_ok=True)
train = read_data(DATA / "train_0irEZ2H.csv", training=True)
test = read_data(DATA / "test_nfaJ3J5.csv")
print(f"Training rows: {len(train):,}; test rows: {len(test):,}")
print(f"History: {train.week.min().date()} to {train.week.max().date()}")
print(f"Forecast: {test.week.min().date()} to {test.week.max().date()}")
display(train.head())

## Explore demand, missingness, and promotions

Dates are explicitly parsed as day/month/year. The single missing transaction price is filled from the same row's base price during feature construction. Store and SKU identifiers are numeric model features; no ordinal interpretation is assumed in the business domain. Tree splits may partition these identifiers, which is a modeling limitation.

In [ ]:
display(train.isna().sum().to_frame("missing"))
display(train.units_sold.describe().to_frame())
weekly = train.groupby("week").units_sold.sum()
fig, ax = plt.subplots(figsize=(12, 3))
weekly.plot(ax=ax, title="Observed weekly demand", ylabel="Units sold")
plt.show()
display(train.groupby(["is_featured_sku", "is_display_sku"]).units_sold.agg(["mean", "median", "count"]))

## Validate on an unseen 12-week period

Hold out the last 12 complete weeks, train only on earlier sales, and predict the entire period from a single origin. Training examples use up to six earlier 12-week blocks, each with at least 26 weeks of history. Historical means and standard deviations (4, 12, 52 weeks) and last observed sales are frozen at each block's origin.

The model is histogram gradient boosting fitted to `log1p(units_sold)`. It predicts nonnegative continuous quantities. Compare RMSLE, MAE, RMSE, and WAPE against the last 12 weeks' mean for each store/SKU. Lower is better. WAPE is a fraction, so 0.25 means 25%. This is one temporal holdout, not a guarantee of performance across seasons.

In [ ]:
report, validation = evaluate(train, horizon=12, iterations=180)
display(pd.DataFrame({"model": report["model"], "recent_mean_baseline": report["recent_mean_baseline"]}))
print({k: v for k, v in report.items() if k not in ["model", "recent_mean_baseline"]})
(OUT / "metrics.json").write_text(json.dumps(report, indent=2) + "\n")
validation.to_csv(OUT / "validation_predictions.csv", index=False)
summary = validation.groupby("week")[["units_sold", "prediction", "baseline"]].sum()
summary.plot(figsize=(12, 3), title="Held-out weekly demand: actual vs forecast", ylabel="Units sold")
plt.show()

In [ ]:
errors = validation.assign(absolute_error=lambda x: abs(x.units_sold - x.prediction))
display(errors.groupby("sku_id").absolute_error.mean().sort_values(ascending=False).head(10).to_frame("MAE"))
print("Model improves RMSLE over baseline:", report["model"]["rmsle"] < report["recent_mean_baseline"]["rmsle"])

## Fit on all observed history and forecast the supplied test rows

Test sales are unknown, so test accuracy cannot be measured. Output preserves test record order and contains exactly `record_ID,units_sold`. The artifact retains history for fixed-origin features. Only load artifacts you trust; joblib uses pickle. Predictions are expected quantities, so they are not rounded.

In [ ]:
model = DemandForecaster(horizon=12, iterations=180).fit(train)
model.save(OUT / "demand_model.joblib")
submission = test[["record_ID"]].copy()
submission["units_sold"] = model.predict(test)
assert submission.record_ID.equals(test.record_ID)
assert np.isfinite(submission.units_sold).all() and (submission.units_sold >= 0).all()
submission.to_csv(OUT / "submission.csv", index=False)
display(submission.head())
print(f"Saved {len(submission):,} forecasts and model in {OUT}")

## Operational use and limitations

Use `demand-forecast evaluate`, `train`, and `predict` for the same workflow outside the notebook; see the README. Retrain when new observed sales arrive. Forecasts must be on the weekly cadence and within the configured horizon after the history cutoff.

Unseen stores or SKUs have missing historical statistics, handled by the model; accuracy for cold starts is not established. Promotions and prices are assumed known. No stockout, availability, returns, or inventory constraints are supplied: observed sales may underestimate unconstrained demand. Monitor per-SKU error, aggregate bias, and drift. Prediction intervals and causal promotion effects are not estimated.